# 项目文本生成示例

这个 notebook 用来演示项目里推荐的文本模型调用方式，并补充当前模型解析信息与多模型测速对比。

推荐入口：
- `text_model.function_TextGeneration`

它负责：
- 读取 `text_model` 用户偏好
- 读取 `data_TextModel.yml`
- 选择对应 provider
- 支持普通回复、流式回复、mini 模型、结构化输出


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    # 从当前目录向上查找，直到定位到包含 Code 目录的仓库根目录。
    for candidate in [start, *start.parents]:
        if (candidate / 'Code').exists():
            return candidate
    raise RuntimeError(f'无法从当前路径定位仓库根目录：{start}')


ROOT = find_repo_root(Path.cwd().resolve())
CODE_DIR = (ROOT / 'Code').resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

TEST_OUTPUT_DIR = ROOT / 'outputs' / 'notebook_tests'
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('项目根目录 =', ROOT)
print('代码目录 =', CODE_DIR)
print('notebook 输出目录 =', TEST_OUTPUT_DIR)


In [ ]:
from text_model.function_TextGeneration import (
    collect_stream_text,
    get_mini_reply,
    get_normal_reply,
    get_selected_text_model_code,
    get_selected_text_model_config,
    get_stream_reply,
    get_structured_reply,
)
from user_info.option_TextModel import ModelRegistry

model_code = get_selected_text_model_code()
model_config = get_selected_text_model_config()
base_url, model_name = model_config.resolve_endpoint()
mini_base_url, mini_model_name = model_config.resolve_endpoint(feature='mini_version')

print('当前选中的文本模型 code =', model_code)
print('配置中的 code =', model_config.code)
print('默认模型名 =', model_name)
print('默认 base_url =', base_url)
print('provider =', model_config.dependence)
print('mini 模型名 =', mini_model_name)
print('mini base_url =', mini_base_url)
print('支持的 feature =', sorted(model_config.features.keys()))


In [ ]:
# 项目里的标准消息格式。
# 你平时主要就是构造这样一个消息列表传给函数。
MESSAGES = [
    {"role": "system", "content": "你是一个简洁的 TRPG 主持人助手。"},
    {"role": "user", "content": "请用三句话描述一座即将被沙暴吞没的古城。"},
]

MESSAGES


## 1. 普通文本回复

这是最常见的用法，会一次性返回完整字符串。


In [ ]:
reply = get_normal_reply(
    input=MESSAGES,
    temperature=0.7,
)

print(reply)


## 2. 流式回复

如果你想模拟前端边生成边显示，可以使用流式接口。


In [ ]:
chunks = list(get_stream_reply(
    input=MESSAGES,
    temperature=0.7,
))

print('分块数量 =', len(chunks))
print('拼接结果 =')
print(collect_stream_text(chunks))


## 3. mini 版本回复

如果当前模型支持 `mini_version`，可以直接走轻量模型。


In [ ]:
mini_reply = get_mini_reply(
    input=MESSAGES,
    temperature=0.7,
)

print(mini_reply)


## 4. 调试：当前模型真实解析结果

这一格专门用来确认当前普通回复和 mini 回复到底打到了哪个模型。


In [ ]:
print('当前偏好 code =', model_config.code)
print('普通回复 ->', model_config.resolve_endpoint())
print('mini 回复 ->', model_config.resolve_endpoint(feature='mini_version'))
print('json_output ->', model_config.resolve_endpoint(feature='json_output'))
print('tool_calls ->', model_config.resolve_endpoint(feature='tool_calls'))


## 5. 多模型测速对比：普通回复 vs mini 回复

这一格会读取 `data_TextModel.yml` 里当前已有的模型，逐个测试普通回复和 mini 回复的耗时。

说明：
- 会真的发起 API 请求，可能消耗额度
- 如果某个模型不支持 mini，会记录为不支持
- 如果某个模型调用失败，会记录错误信息


In [ ]:
import json
import time
from pathlib import Path

registry = ModelRegistry.load(CODE_DIR / 'data' / 'data_TextModel.yml')
compare_messages = [
    {"role": "system", "content": "你是一个简洁的助手。"},
    {"role": "user", "content": "请用两句话描述一座古老而危险的沙漠遗迹。"},
]


def make_pref_file(model_code: str) -> Path:
    pref_path = TEST_OUTPUT_DIR / f'text_pref_{model_code}.json'
    pref_path.write_text(
        json.dumps({"language": "zh-CN", "text_model": {"code": model_code}}, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    return pref_path


def time_call(fn, **kwargs):
    start = time.perf_counter()
    result = fn(**kwargs)
    elapsed = time.perf_counter() - start
    return elapsed, result


results = []
for model in registry.list_models():
    pref_path = make_pref_file(model.code)
    row = {
        'model_code': model.code,
        'model_name': model.name,
        'normal_seconds': None,
        'mini_seconds': None,
        'normal_status': 'pending',
        'mini_status': 'pending',
    }

    try:
        elapsed, text = time_call(
            get_normal_reply,
            input=compare_messages,
            preference_path=pref_path,
            temperature=0.7,
        )
        row['normal_seconds'] = round(elapsed, 2)
        row['normal_status'] = f"ok: {text[:40]}"
    except Exception as exc:
        row['normal_status'] = f'error: {exc}'

    try:
        if not model.features['mini_version'].supported:
            raise ValueError('not supported')

        elapsed, text = time_call(
            get_mini_reply,
            input=compare_messages,
            preference_path=pref_path,
            temperature=0.7,
        )
        row['mini_seconds'] = round(elapsed, 2)
        row['mini_status'] = f"ok: {text[:40]}"
    except Exception as exc:
        row['mini_status'] = f'error: {exc}'

    results.append(row)

results


In [ ]:
try:
    import pandas as pd
    display(pd.DataFrame(results))
except Exception:
    for row in results:
        print(row)


## 6. 结构化输出

这个适合把模型输出约束成固定字段，方便程序继续处理。


In [ ]:
from pydantic import BaseModel, Field


class CitySummary(BaseModel):
    名称: str = Field(description='古城名称')
    氛围: str = Field(description='整体氛围描述')
    危险: list[str] = Field(description='当前主要危险列表')


structured = get_structured_reply(
    input=[
        {"role": "system", "content": "请严格根据 schema 输出。"},
        {"role": "user", "content": "输出一个即将被沙暴吞没的古城摘要。"},
    ],
    output_schema=CitySummary,
)

print(structured)
print(structured.model_dump())


## 说明

以后在项目代码里，优先从下面这个文件导入：

`Code/text_model/function_TextGeneration.py`

现在项目推荐统一从 `function_TextGeneration.py` 导入，旧的 `function_TextModel` 兼容入口已经不再使用。
